# Step 5 - Drift and quality monitoring with Evidently AI

**CSC3119 Assignment 4.** A model is trained on the past and used on the future. This notebook measures whether the future looks like the past, using **Evidently AI**, and puts a retraining trigger in writing.

Two experiments:

* **Run A - operational monitoring.** Reference = the training hours (with **out-of-fold** predictions, so the reference error is honest); current = the hold-out hours. Weather and hour columns drift because the hold-out contains partial days, but the target and the error barely move - the model is still fit for purpose.
* **Run B - seasonal drift case study.** January vs June. The pollution level collapses between the two, so a model trained only in January would fail in June. This is the drift a retraining policy must catch.

Evidently is wrapped in `try/except`: a version mismatch on RunPod cannot break the pipeline, as the brief requires. A small scipy drift table is also written so the numbers can go straight into the report.

In [1]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
from pathlib import Path
import sys, json, time, warnings, traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.base import clone
from sklearn.model_selection import GroupKFold, cross_val_predict

warnings.filterwarnings("ignore")
PROJECT = Path.cwd()
while not (PROJECT / "src" / "aq_common.py").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "src"))
import aq_common as A

np.random.seed(A.SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 185)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

A.set_stage("monitor")
A.banner("STEP 5 / 6  -  Drift monitoring with Evidently AI")
LOG = A.PipelineLog("step5_monitoring")


  STEP 5 / 6  -  Drift monitoring with Evidently AI
2026-10-18 12:35:22 | INFO | === step5_monitoring ===


## 0. Rebuild the monitoring frames
Reference predictions are produced with `cross_val_predict` over the same 6-hour blocked folds used in Step 3, so no reference hour is scored by a model that saw it.

In [2]:
B = joblib.load(A.stage_dir("model") / "best_pm25_model.joblib")
grid = pd.read_parquet(A.stage_dir("prep") / "hourly_clean.parquet")
ZP = {"mean": B["pm25_mean"], "std": B["pm25_std"]}
FEATS = B["features"]

df = grid[grid["pm25_filled"].notna()].copy()
df = A.add_calendar_features(df, "datetime")
df["y"] = A.forward_z(df["pm25_filled"], ZP)
df = df.sort_values("datetime").reset_index(drop=True)
df["window"] = df["datetime"].dt.to_period("M")
first = [pd.Timestamp(d).normalize() for d in sorted(df.groupby("window")["datetime"].min())]
is_tr = df["datetime"].dt.normalize().isin(first)
train = df[is_tr].reset_index(drop=True)
test = df[~is_tr].reset_index(drop=True)

def blocK(dates):
    d = pd.to_datetime(dates)
    return (d.dt.dayofyear * 4 + (d.dt.hour // 6)).astype(int)

groups = blocK(train["datetime"])
cv = GroupKFold(n_splits=5)
print(f"train {len(train):,} | test {len(test):,} | Jan {int((df.window==df.window.min()).sum()):,}")
LOG.log(f"monitoring frame train={len(train)} test={len(test)}")

# The reference predictions MUST be out-of-fold: each training hour is predicted by
# a model that did not train on it.  In-sample predictions would be unrealistically
# accurate and every later comparison would look like drift.
oof = cross_val_predict(clone(B["model"]), train[FEATS], train["y"].to_numpy(),
                        groups=groups, cv=cv, n_jobs=-1)
train = train.assign(prediction=oof)
test = test.assign(prediction=B["model"].predict(test[FEATS]))
print("reference (train) predictions are out-of-fold; current (test) use the shipped model")

train 6,804 | test 9,444 | Jan 5,693
2026-10-18 12:35:24 | INFO | monitoring frame train=6804 test=9444


reference (train) predictions are out-of-fold; current (test) use the shipped model


In [3]:
WATCH = ["temp", "humidity", "latitude", "longitude", "hour_sin", "hour_cos",
         "target", "prediction"]

# Evidently wants a target and a prediction column; name them explicitly so the
# regression preset knows what to score.
def frame(features_df):
    return features_df.rename(columns={"y": "target"})[WATCH]

ref_A, cur_A = frame(train), frame(test)
jan = df[df["window"] == df["window"].min()].copy()
jun = df[df["window"] == df["window"].max()].copy()
jan["prediction"] = B["model"].predict(jan[FEATS])
jun["prediction"] = B["model"].predict(jun[FEATS])
ref_B, cur_B = frame(jan), frame(jun)
print(f"Run A  reference {len(ref_A):,} -> current {len(cur_A):,}")
print(f"Run B  January   {len(ref_B):,} -> June      {len(cur_B):,}")

Run A  reference 6,804 -> current 9,444
Run B  January   5,693 -> June      5,282


## 1. A drift table you can quote
Wasserstein distance (normalised by the reference standard deviation) and the Kolmogorov-Smirnov p-value, computed directly with scipy. Two independent statistics agreeing is stronger evidence than one.

In [4]:
print("\n" + "="*78)
print(" 1. DRIFT TABLE (Evidently-independent, computed with scipy)")
print("="*78)

# Evidently's numbers are inside its HTML; to put drift in the report and the study
# guide as a table we also compute the two statistics ourselves.  Wasserstein =
# how far one distribution must move to match the other, divided by the reference
# standard deviation (0 = identical, > 0.1 = drift).  K-S p-value < 0.05 = drift.
def drift_table(reference, current, cols=WATCH):
    rows = []
    for c in cols:
        a = reference[c].dropna().to_numpy()
        b = current[c].dropna().to_numpy()
        sd = a.std() if a.std() > 0 else 1.0
        wd = wasserstein_distance(a, b) / sd
        ks = ks_2samp(a, b).pvalue
        rows.append({"column": c, "wasserstein_normed": round(wd, 3),
                     "ks_pvalue": ks, "drift": bool(wd > 0.1)})
    return pd.DataFrame(rows)

drift_A = drift_table(ref_A, cur_A)
drift_B = drift_table(ref_B, cur_B)
for name, t in [("RUN A  train vs hold-out", drift_A), ("RUN B  January vs June", drift_B)]:
    share = t["drift"].mean()
    print(f"\n{name}:  {int(t['drift'].sum())}/{len(t)} columns drifted (share {share:.2f})")
    print(t.to_string(index=False))
drift_A.to_csv(A.stage_dir("monitor") / "drift_table_runA.csv", index=False)
drift_B.to_csv(A.stage_dir("monitor") / "drift_table_runB.csv", index=False)
print("\nDataset-level drift is declared when the drifted share >= 0.5.")


 1. DRIFT TABLE (Evidently-independent, computed with scipy)



RUN A  train vs hold-out:  5/8 columns drifted (share 0.62)
    column  wasserstein_normed     ks_pvalue  drift
      temp               0.160  2.144037e-30   True
  humidity               0.261  5.961209e-47   True
  latitude               0.009  9.769628e-01  False
 longitude               0.005  9.057952e-01  False
  hour_sin               0.216  1.298752e-92   True
  hour_cos               0.402 1.156930e-120   True
    target               0.085  2.386156e-10  False
prediction               0.137  2.899580e-42   True

RUN B  January vs June:  4/8 columns drifted (share 0.50)
    column  wasserstein_normed     ks_pvalue  drift
      temp               0.368  5.040281e-52   True
  humidity               0.588 6.689619e-108   True
  latitude               0.049  3.426176e-13  False
 longitude               0.098  8.913610e-14  False
  hour_sin               0.005  9.999680e-01  False
  hour_cos               0.016  8.945140e-01  False
    target               0.787  0.000000e+00   T

## 2. The Evidently reports
Three HTML reports are written to `outputs/05_monitor/`: input drift plus regression quality for Run A, a quality-only report, and the January-vs-June seasonal comparison.

In [5]:
print("\n" + "="*78)
print(" 2. EVIDENTLY REPORTS")
print("="*78)

# The whole Evidently section is wrapped so a version mismatch on RunPod cannot
# break the pipeline, exactly as the brief requires.
EVI = {"available": False, "reports": {}, "metrics": {}}
try:
    from evidently import Report, Dataset, DataDefinition, Regression
    from evidently.presets import DataDriftPreset, RegressionPreset
    dd = DataDefinition(regression=[Regression(target="target", prediction="prediction")])

    def run_report(metrics, ref, cur, filename):
        t0 = time.time()
        report = Report(metrics)
        snap = report.run(reference_data=Dataset.from_pandas(ref, data_definition=dd),
                          current_data=Dataset.from_pandas(cur, data_definition=dd))
        out = A.stage_dir("monitor") / filename
        snap.save_html(str(out))
        print(f"  wrote {filename}  ({time.time()-t0:.1f}s)")
        return snap

    snapA = run_report([DataDriftPreset(), RegressionPreset()], ref_A, cur_A,
                       "evidently_data_drift.html")
    snapM = run_report([RegressionPreset()], ref_A, cur_A,
                       "evidently_model_quality.html")
    snapB = run_report([DataDriftPreset(), RegressionPreset()], ref_B, cur_B,
                       "evidently_seasonal_drift_jan_vs_jun.html")

    def pull(snap, keys):
        out = {}
        for m in snap.dict()["metrics"]:
            name = str(m.get("metric_name") or m.get("config", {}).get("type", ""))
            for k in keys:
                if name.startswith(k):
                    out[name] = m.get("value")
        return out

    EVI["available"] = True
    EVI["reports"] = {f: str(A.stage_dir("monitor") / f) for f in
                      ["evidently_data_drift.html", "evidently_model_quality.html",
                       "evidently_seasonal_drift_jan_vs_jun.html"]}
    EVI["metrics"] = {"runA_drift": pull(snapA, ["DriftedColumnsCount"]),
                      "runA_quality": pull(snapM, ["RMSE", "MAE", "MeanError", "R2Score"]),
                      "runB_drift": pull(snapB, ["DriftedColumnsCount"])}
    print("\nEvidently summary metrics:", json.dumps(EVI["metrics"], default=str)[:400])
    LOG.log("Evidently reports written: data_drift, model_quality, seasonal")
except Exception as exc:
    print("Evidently failed and was skipped (pipeline continues):", repr(exc))
    traceback.print_exc()
    LOG.log(f"Evidently skipped: {exc!r}")


 2. EVIDENTLY REPORTS


  wrote evidently_data_drift.html  (5.1s)


  wrote evidently_model_quality.html  (1.1s)


  wrote evidently_seasonal_drift_jan_vs_jun.html  (2.5s)

Evidently summary metrics: {"runA_drift": {"DriftedColumnsCount(drift_share=0.5)": {"count": 5.0, "share": 0.625}}, "runA_quality": {"MeanError(regression_name=default,error_plot=True,error_distr=False,error_normality=False)": {"mean": -0.01781963795881961, "std": 0.642425375207848}, "RMSE(regression_name=default,error_plot=False,error_distr=True,error_normality=False)": 0.6426384686642502, "MAE(regression_name=default,erro
2026-10-18 12:36:14 | INFO | Evidently reports written: data_drift, model_quality, seasonal


In [6]:
print("\n" + "="*78)
print(" 3. DRIFT FIGURE")
print("="*78)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
for ax, t, title in [(axes[0], drift_A, "Run A: train vs hold-out"),
                     (axes[1], drift_B, "Run B: January vs June")]:
    s = t.sort_values("wasserstein_normed")
    colours = ["#c44e52" if d else "#55a868" for d in s["drift"]]
    ax.barh(s["column"], s["wasserstein_normed"], color=colours)
    ax.axvline(0.1, color="k", ls="--", lw=1, label="drift threshold 0.1")
    ax.set_xlabel("Wasserstein / reference sd")
    ax.set_title(title)
    ax.legend(fontsize=7)
fig.suptitle("Fig 5.1  Feature and prediction drift", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.96])
A.savefig(fig, "fig5_1_drift.png")

fig, ax = plt.subplots(figsize=(9, 4.4))
pooled = pd.concat([t.assign(run=n) for t, n in
                    [(drift_A, "Run A"), (drift_B, "Run B")]])
for run, g in pooled.groupby("run"):
    ax.scatter(g["wasserstein_normed"], g["ks_pvalue"].clip(1e-300, 1), s=60, label=run)
    for _, r in g.iterrows():
        ax.annotate(r["column"], (r["wasserstein_normed"], max(r["ks_pvalue"], 1e-300)),
                    fontsize=6, textcoords="offset points", xytext=(4, 3))
ax.axvline(0.1, color="k", ls="--", lw=1); ax.axhline(0.05, color="#c44e52", ls=":", lw=1)
ax.set_yscale("log")
ax.set_xlabel("Wasserstein / reference sd"); ax.set_ylabel("K-S p-value (log)")
ax.set_title("Fig 5.2  Two drift statistics agree (top-right = strong drift)")
ax.legend(fontsize=7)
fig.tight_layout()
A.savefig(fig, "fig5_2_drift_scatter.png")
print("Red columns in Fig 5.1 crossed the Wasserstein threshold. In Fig 5.2 points")
print("in the top-right quadrant moved far AND significantly.")


 3. DRIFT FIGURE


Red columns in Fig 5.1 crossed the Wasserstein threshold. In Fig 5.2 points
in the top-right quadrant moved far AND significantly.


## 3. What the numbers mean, and when to retrain

In [7]:
share_A = float(drift_A["drift"].mean()); share_B = float(drift_B["drift"].mean())
meta = {
    "evidently_available": EVI["available"],
    "runA": {"reference": "training hours (out-of-fold predictions)",
             "current": "hold-out hours", "drift_share": share_A,
             "dataset_drift": bool(share_A >= 0.5),
             "drifted_columns": drift_A.loc[drift_A["drift"], "column"].tolist(),
             "table": drift_A.to_dict(orient="records"),
             "evidently": EVI["metrics"].get("runA_quality", {})},
    "runB": {"reference": "January", "current": "June", "drift_share": share_B,
             "dataset_drift": bool(share_B >= 0.5),
             "drifted_columns": drift_B.loc[drift_B["drift"], "column"].tolist(),
             "table": drift_B.to_dict(orient="records")},
    "reports": EVI["reports"],
    "reading": ("Run A: the hold-out includes partial days, so some hour/weather "
                "columns drift statistically but the target and error stay stable - "
                "no retraining needed. Run B: January is far more polluted than June; "
                "a model trained only in January fails in June, which is why we train "
                "across every window and monitor continually."),
    "retrain_trigger": ("Retrain when target or prediction drift appears (Wasserstein "
                        "> 0.1) AND RMSE on fresh labelled data rises meaningfully "
                        "above the reference; keep versioned models to roll back."),
    "env": A.env_facts(),
}
A.save_json(meta, A.stage_dir("monitor") / "monitoring_meta.json")
LOG.log(f"drift share Run A={share_A:.2f} Run B={share_B:.2f}")
LOG.save({"stage": "monitoring", "runA_drift_share": share_A, "runB_drift_share": share_B,
          "evidently": EVI["available"]})

print("\n" + "="*78)
print("  STEP 5 COMPLETE")
print("="*78)
for f in sorted(A.stage_dir("monitor").glob("*")):
    print(f"  {f.name:46s} {f.stat().st_size:>12,d} bytes")
print("\nNext:  Step6_Deployment.ipynb (RunPod packaging + serving contract)")

2026-10-18 12:36:17 | INFO | drift share Run A=0.62 Run B=0.50

  STEP 5 COMPLETE
  drift_table_runA.csv                                    381 bytes
  drift_table_runB.csv                                    348 bytes
  evidently_data_drift.html                         4,338,335 bytes
  evidently_model_quality.html                      3,775,114 bytes
  evidently_seasonal_drift_jan_vs_jun.html          4,338,389 bytes
  fig5_1_drift.png                                     55,627 bytes
  fig5_2_drift_scatter.png                             57,946 bytes
  monitoring_meta.json                                  5,112 bytes

Next:  Step6_Deployment.ipynb (RunPod packaging + serving contract)
